# Preprocesamiento y representación de texto

**Duración estimada:** 45 minutos

## Objetivo
Hasta ahora trabajaste con spaCy para tokenizar, lematizar, etiquetar y reconocer entidades. Todas esas operaciones producen *información sobre el texto*, pero todavía no decidimos **cómo convertir un texto en una representación estructurada** que sirva como entrada para análisis posteriores.

En este cuaderno vas a:
- recorrer paso a paso las decisiones de preprocesamiento que transforman texto crudo en una lista de tokens;
- comparar distintos pipelines de preprocesamiento y observar cómo cada uno altera los resultados;
- ver en la práctica cómo otra herramienta (Stanza) produce resultados distintos sobre el mismo texto;
- entender que la representación final no es neutral: cada decisión (lematizar o no, filtrar stopwords o no, qué herramienta usar) descarta información y puede introducir sesgos.

## Qué NO cubre este cuaderno
Todavía no vamos a desarrollar TF-IDF ni ningún esquema de ponderación numérica. Eso se trabaja en el próximo cuaderno. Pero para que la mención no quede en el aire, una intuición rápida:

**TF-IDF** (*Term Frequency – Inverse Document Frequency*) es un método para asignarle un **peso numérico** a cada palabra en un documento. La idea básica es: una palabra es importante para un documento si aparece mucho en *ese* documento (TF alta) pero poco en el *resto* de los documentos (IDF alta). Así, palabras como "de" o "la" — que aparecen en todos lados — reciben peso bajo, mientras que términos específicos del tema reciben peso alto.

Para que TF-IDF funcione, necesita recibir una **lista de tokens** por cada documento. Esa lista es exactamente lo que vamos a aprender a construir acá. El foco de este cuaderno está en **qué le damos de comer** a esos métodos.

## Relación con los cuadernos anteriores
- En `001` a `003` construiste la base con el pipeline estándar de spaCy.
- En `004` diagnosticaste fallas del modelo frente al español rioplatense.
- En `005` integraste todo en un analizador de noticias.
- Acá vas a formalizar el paso que conecta el texto crudo con cualquier análisis cuantitativo posterior.

---
## 1. Carga de spaCy y texto de ejemplo

In [ ]:
import spacy
from collections import Counter

nlp = spacy.load("es_core_news_sm")
print(f"Modelo cargado: {nlp.meta['name']}")
print(f"Pipeline activo: {nlp.pipe_names}")

In [ ]:
# Texto de ejemplo en español rioplatense
texto = (
    "Vos sabés que si querés venís mañana a la clase de NLP. "
    "La clase es sobre procesamiento de texto y representación de datos. "
    "Tenés que traer la notebook con spaCy instalado. "
    "Si no podés mañana, avisá y coordinamos otro día. "
    "El procesamiento de lenguaje natural es clave para entender datos textuales."
)

doc = nlp(texto)
print(f"Texto procesado: {len(doc)} tokens")
print(f"Oraciones detectadas: {len(list(doc.sents))}")

---
## 2. Extracción de representaciones básicas

Antes de decidir qué pipeline usar, necesitamos ver qué información nos da spaCy para cada token. Vamos a extraer cuatro vistas distintas del mismo texto:

1. **Tokens originales** — el texto tal como aparece.
2. **Tokens en minúscula** — normalización básica.
3. **Lemas** — forma canónica según el modelo.
4. **Tokens filtrados** — sin stopwords ni puntuación.

In [ ]:
# 2.1 Tokens originales
tokens_originales = [token.text for token in doc]
print("TOKENS ORIGINALES:")
print(tokens_originales)
print(f"Total: {len(tokens_originales)}")

In [ ]:
# 2.2 Tokens en minúscula
tokens_lower = [token.text.lower() for token in doc]
print("TOKENS LOWERCASED:")
print(tokens_lower)
print(f"Total: {len(tokens_lower)}")

In [ ]:
# 2.3 Lemas
lemas = [token.lemma_ for token in doc]
print("LEMAS:")
print(lemas)
print(f"Total: {len(lemas)}")

In [ ]:
# 2.4 Tokens filtrados (solo alfabéticos, sin stopwords)
tokens_filtrados = [
    token.text for token in doc
    if token.is_alpha and not token.is_stop
]
print("TOKENS FILTRADOS (alfabéticos, sin stopwords):")
print(tokens_filtrados)
print(f"Total: {len(tokens_filtrados)} (de {len(tokens_originales)} originales)")

Observá las diferencias:
- Pasar a *lowercase* unifica mayúsculas y minúsculas, pero pierde información sobre nombres propios o siglas.
- Los lemas reducen variantes flexivas a una forma base, pero dependen de la calidad del modelo (recordá lo que vimos en el cuaderno `004` sobre el voseo).
- El filtrado elimina ruido, pero también descarta palabras funcionales que pueden tener valor en ciertos análisis.

---
## 3. Comparación de pipelines de preprocesamiento

Ahora vamos a definir tres estrategias distintas de preprocesamiento y comparar sus resultados sobre el mismo texto. Cada pipeline es una función que recibe un `Doc` de spaCy y devuelve una lista de strings.

### Pipeline A: naive (lowercase + solo alfabéticos)

La estrategia más simple: pasar todo a minúscula y conservar solo tokens que contengan letras. No usa información lingüística del modelo.

In [ ]:
def pipeline_a(doc):
    """Pipeline naive: lowercase + solo palabras alfabéticas."""
    return [
        token.text.lower()
        for token in doc
        if token.is_alpha
    ]

resultado_a = pipeline_a(doc)
print("PIPELINE A (naive):")
print(resultado_a)
print(f"\nTotal tokens: {len(resultado_a)}")

freq_a = Counter(resultado_a)
print("\nTop 10 palabras más frecuentes:")
for palabra, cuenta in freq_a.most_common(10):
    print(f"  {palabra:20} -> {cuenta}")

### Pipeline B: lingüístico (lemas + sin stopwords)

Usa la información del modelo para lematizar y eliminar stopwords. Es la estrategia más habitual en tutoriales de NLP.

In [ ]:
def pipeline_b(doc):
    """Pipeline lingüístico: lemas, sin stopwords, solo alfabéticos."""
    return [
        token.lemma_.lower()
        for token in doc
        if token.is_alpha and not token.is_stop
    ]

resultado_b = pipeline_b(doc)
print("PIPELINE B (lingüístico):")
print(resultado_b)
print(f"\nTotal tokens: {len(resultado_b)}")

freq_b = Counter(resultado_b)
print("\nTop 10 palabras más frecuentes:")
for palabra, cuenta in freq_b.most_common(10):
    print(f"  {palabra:20} -> {cuenta}")

### Pipeline C: ajustado (lemas corregidos para voseo)

Igual al B, pero con un diccionario de corrección manual para formas del voseo que spaCy no lematiza bien. Esto retoma lo que diagnosticamos en el cuaderno `004`.

In [ ]:
# Diccionario de correcciones para el voseo
CORRECCIONES_VOSEO = {
    "sos": "ser",
    "tenés": "tener",
    "sabés": "saber",
    "querés": "querer",
    "venís": "venir",
    "podés": "poder",
    "avisá": "avisar",
    "revisá": "revisar",
    "analizá": "analizar",
}


def pipeline_c(doc, correcciones=None):
    """Pipeline ajustado: lemas corregidos, sin stopwords, solo alfabéticos."""
    if correcciones is None:
        correcciones = {}

    resultado = []
    for token in doc:
        if not token.is_alpha or token.is_stop:
            continue

        forma = token.text.lower()
        # Si existe corrección manual, usarla; si no, usar el lema del modelo
        if forma in correcciones:
            lema = correcciones[forma]
        else:
            lema = token.lemma_.lower()

        resultado.append(lema)

    return resultado


resultado_c = pipeline_c(doc, CORRECCIONES_VOSEO)
print("PIPELINE C (ajustado):")
print(resultado_c)
print(f"\nTotal tokens: {len(resultado_c)}")

freq_c = Counter(resultado_c)
print("\nTop 10 palabras más frecuentes:")
for palabra, cuenta in freq_c.most_common(10):
    print(f"  {palabra:20} -> {cuenta}")

### Comparación lado a lado

In [ ]:
print("=" * 60)
print("COMPARACIÓN DE PIPELINES")
print("=" * 60)

pipelines = {
    "A (naive)": resultado_a,
    "B (lingüístico)": resultado_b,
    "C (ajustado)": resultado_c,
}

for nombre, tokens in pipelines.items():
    freq = Counter(tokens)
    print(f"\n--- {nombre} ---")
    print(f"  Tokens totales:  {len(tokens)}")
    print(f"  Tokens únicos:   {len(set(tokens))}")
    print(f"  Top 5: {freq.most_common(5)}")

---
## 4. Análisis: qué cambia entre pipelines

Tomate un minuto para reflexionar sobre las diferencias. Estas preguntas te ayudan a organizar la observación:

1. **¿Cambian los resultados entre pipelines?** Compará la cantidad de tokens totales y únicos. ¿Qué pipeline produce la lista más corta? ¿Por qué?

2. **¿Qué información se pierde en cada paso?**
   - El Pipeline A conserva stopwords ("de", "la", "que"), que dominan el ranking de frecuencia pero aportan poca información sobre el contenido.
   - El Pipeline B elimina stopwords, pero los lemas pueden ser incorrectos para formas del voseo.
   - El Pipeline C corrige esos lemas, pero depende de un diccionario construido a mano.

3. **¿Qué representación parece más útil?** Depende de la tarea. Para un conteo de frecuencias orientado al contenido, el Pipeline C da resultados más limpios. Pero si la tarea es estudiar estructura gramatical, eliminar stopwords puede ser contraproducente.

No existe un pipeline universalmente correcto. Cada decisión de preprocesamiento es una hipótesis sobre qué importa y qué no.

---
## 5. Propagación de errores: cómo un lema incorrecto altera los resultados

En el cuaderno `004` vimos que spaCy no lematiza bien las formas del voseo. Acá vamos a medir el impacto concreto de ese error sobre las frecuencias.

In [ ]:
# Texto con varias formas verbales del voseo
texto_voseo = (
    "Si vos querés, podés venir mañana. Pero si no podés, avisá. "
    "Yo sé que vos sabés de qué se trata. Vos siempre sabés. "
    "Además podés traer lo que quieras. Si querés, venís."
)

doc_voseo = nlp(texto_voseo)

# Pipeline B: lemas directos de spaCy (sin corrección)
lemas_spacy = [
    token.lemma_.lower()
    for token in doc_voseo
    if token.is_alpha and not token.is_stop
]

# Pipeline C: lemas corregidos
lemas_corregidos = pipeline_c(doc_voseo, CORRECCIONES_VOSEO)

print("--- Frecuencias con lemas de spaCy (Pipeline B) ---")
freq_spacy = Counter(lemas_spacy)
for palabra, cuenta in freq_spacy.most_common(10):
    print(f"  {palabra:20} -> {cuenta}")

print("\n--- Frecuencias con lemas corregidos (Pipeline C) ---")
freq_corregido = Counter(lemas_corregidos)
for palabra, cuenta in freq_corregido.most_common(10):
    print(f"  {palabra:20} -> {cuenta}")

In [ ]:
# Comparación token a token para ver dónde difieren
print("--- Detalle: dónde difieren los lemas ---")
print(f"{'TOKEN':<15} {'LEMA SPACY':<20} {'LEMA CORREGIDO':<20} {'¿DIFIERE?'}")
print("-" * 70)

for token in doc_voseo:
    if not token.is_alpha or token.is_stop:
        continue

    lema_spacy = token.lemma_.lower()
    forma = token.text.lower()
    lema_manual = CORRECCIONES_VOSEO.get(forma, lema_spacy)

    difiere = "<-- SÍ" if lema_spacy != lema_manual else ""
    print(f"  {token.text:<15} {lema_spacy:<20} {lema_manual:<20} {difiere}")

Observá cómo el error de lematización se **propaga** al conteo de frecuencias:
- Sin corrección, formas como "querés", "podés" y "sabés" aparecen con lemas incorrectos y se dispersan en el conteo.
- Con corrección, se agrupan correctamente bajo "querer", "poder" y "saber".

Este efecto se amplifica en corpus grandes. Un error de lema que parece menor en una oración se convierte en un sesgo sistemático cuando se repite miles de veces.

---
## 6. Comparación práctica: spaCy vs. Stanza

Hasta acá usamos exclusivamente spaCy. Pero no es la única herramienta que puede tokenizar y lematizar español. **Stanza**, desarrollada por el Stanford NLP Group, es otra librería que ofrece análisis morfológico con modelos entrenados sobre corpus distintos.

¿Por qué importa comparar? Porque si dos herramientas producen lemas diferentes para el mismo texto, la representación final cambia, y con ella los resultados de cualquier análisis posterior. Vamos a verlo en la práctica.

In [ ]:
# Instalación de Stanza y spacy-stanza (ejecutar solo la primera vez)
!pip install stanza spacy-stanza -q

In [ ]:
import stanza

# Descarga del modelo en español (solo la primera vez, puede tardar un minuto)
stanza.download("es", verbose=False)

# Carga del pipeline de Stanza
# Especificamos solo los procesadores que necesitamos para evitar
# problemas de compatibilidad con procesadores más pesados.
nlp_stanza = stanza.Pipeline("es", processors="tokenize,mwt,pos,lemma", verbose=False)
print("Modelo de Stanza cargado.")

### 6.1 Comparación de lemas: spaCy vs. Stanza

Procesamos el mismo texto con ambas herramientas y comparamos los lemas que asigna cada una. Prestá especial atención a las formas del voseo.

In [ ]:
# Texto de prueba con formas del voseo
texto_comparacion = "Vos sabés que si querés podés venir. Avisá si no venís."

# Procesamiento con spaCy
doc_spacy = nlp(texto_comparacion)

# Procesamiento con Stanza
doc_stanza = nlp_stanza(texto_comparacion)

# Resultados de spaCy
print("=" * 65)
print("spaCy")
print("=" * 65)
print(f"{'TOKEN':<15} {'LEMA':<15} {'POS':<10}")
print("-" * 40)
for token in doc_spacy:
    if not token.is_punct and not token.is_space:
        print(f"  {token.text:<15} {token.lemma_:<15} {token.pos_:<10}")

# Resultados de Stanza
print(f"\n{'=' * 65}")
print("Stanza")
print("=" * 65)
print(f"{'TOKEN':<15} {'LEMA':<15} {'POS':<10}")
print("-" * 40)
for oracion in doc_stanza.sentences:
    for token in oracion.words:
        if token.upos != "PUNCT":
            print(f"  {token.text:<15} {token.lemma:<15} {token.upos:<10}")

### 6.2 Comparación lado a lado

Para ver las diferencias con más claridad, armemos una tabla que ponga los lemas de ambas herramientas uno al lado del otro.

In [ ]:
# Extraemos lemas de spaCy (sin puntuación)
lemas_sp = [
    (token.text, token.lemma_, token.pos_)
    for token in doc_spacy
    if not token.is_punct and not token.is_space
]

# Extraemos lemas de Stanza (sin puntuación)
lemas_st = [
    (word.text, word.lemma, word.upos)
    for sent in doc_stanza.sentences
    for word in sent.words
    if word.upos != "PUNCT"
]

print(f"{'TOKEN':<12} {'LEMA spaCy':<15} {'LEMA Stanza':<15} {'¿DIFIERE?'}")
print("-" * 55)

# Comparamos posición a posición
for i in range(min(len(lemas_sp), len(lemas_st))):
    texto_sp, lema_sp, _ = lemas_sp[i]
    texto_st, lema_st, _ = lemas_st[i]

    difiere = "<-- SÍ" if lema_sp.lower() != lema_st.lower() else ""
    print(f"  {texto_sp:<12} {lema_sp:<15} {lema_st:<15} {difiere}")

### 6.3 Impacto en un pipeline de preprocesamiento

Veamos cómo esas diferencias de lematización se traducen en listas de tokens distintas cuando aplicamos un pipeline tipo B (lemas + sin stopwords).

In [ ]:
# Pipeline B con spaCy
tokens_spacy = [
    token.lemma_.lower()
    for token in doc_spacy
    if token.is_alpha and not token.is_stop
]

# Pipeline equivalente con Stanza
# Stanza no tiene is_stop incorporado, así que usamos la lista de spaCy
stopwords_es = nlp.Defaults.stop_words

tokens_stanza = [
    word.lemma.lower()
    for sent in doc_stanza.sentences
    for word in sent.words
    if word.upos != "PUNCT"
    and word.text.isalpha()
    and word.text.lower() not in stopwords_es
]

print("Pipeline B con spaCy:")
print(f"  Tokens: {tokens_spacy}")
print(f"  Únicos: {len(set(tokens_spacy))}")

print("\nPipeline B con Stanza:")
print(f"  Tokens: {tokens_stanza}")
print(f"  Únicos: {len(set(tokens_stanza))}")

# Diferencias
set_spacy = set(tokens_spacy)
set_stanza = set(tokens_stanza)

solo_spacy = set_spacy - set_stanza
solo_stanza = set_stanza - set_spacy

if solo_spacy or solo_stanza:
    print("\nDiferencias:")
    if solo_spacy:
        print(f"  Solo en spaCy:  {solo_spacy}")
    if solo_stanza:
        print(f"  Solo en Stanza: {solo_stanza}")
else:
    print("\nAmbas herramientas produjeron los mismos tokens únicos.")

### 6.4 Integración: usar Stanza desde la API de spaCy

Si después de comparar decidís que Stanza lematiza mejor para tu caso, no hace falta reescribir todo tu código. La librería `spacy-stanza` permite usar modelos de Stanza directamente desde la interfaz de spaCy. Así conservás todo tu pipeline (filtros, `Matcher`, `EntityRuler`) pero con un motor de análisis distinto por debajo.

**Nota técnica:** al cargar el pipeline híbrido, es importante especificar solo los procesadores que necesitamos (`tokenize,mwt,pos,lemma`). Si se omite este parámetro, `spacy-stanza` intenta cargar todos los procesadores de Stanza (incluido el de *constituency parsing*), lo que puede generar errores de compatibilidad entre versiones.

In [ ]:
import spacy_stanza

# Carga el modelo de Stanza con la interfaz de spaCy.
# Limitamos los procesadores para evitar errores de compatibilidad.
nlp_hibrido = spacy_stanza.load_pipeline(
    "es",
    processors="tokenize,mwt,pos,lemma",
    verbose=False,
)
print(f"Pipeline híbrido cargado: {nlp_hibrido.pipe_names}")

In [ ]:
# Procesamos el mismo texto con el pipeline híbrido
doc_hibrido = nlp_hibrido(texto_comparacion)

print("--- Pipeline híbrido (Stanza vía spaCy) ---")
print(f"{'TOKEN':<15} {'LEMA':<15} {'POS':<10} {'is_stop'}")
print("-" * 50)
for token in doc_hibrido:
    if not token.is_punct and not token.is_space:
        print(f"  {token.text:<15} {token.lemma_:<15} {token.pos_:<10} {token.is_stop}")

Observá que el pipeline híbrido se usa **exactamente igual** que spaCy puro: `token.text`, `token.lemma_`, `token.pos_`, `token.is_stop`. Esto significa que podés intercambiar el motor de análisis sin tocar el resto de tu código.

Esta es una ventaja arquitectónica importante: **separar la interfaz del motor** permite experimentar con distintas herramientas sin reescribir pipelines completos.

### 6.5 Resumen de la comparación

| Aspecto | spaCy (`es_core_news_sm`) | Stanza (`es`) |
|---|---|---|
| **Corpus de entrenamiento** | AnCora (predominantemente peninsular) | AnCora + UD Spanish (varias fuentes) |
| **Voseo** | Lematización frecuentemente incorrecta | Puede manejar mejor algunas formas |
| **Stopwords** | Lista incorporada (`token.is_stop`) | No incluye lista propia |
| **Velocidad** | Más rápido en producción | Más lento (modelos neuronales más pesados) |
| **Integración** | API nativa muy completa | Integrable vía `spacy-stanza` |

La conclusión no es que una herramienta sea mejor que la otra. Es que **la elección de herramienta es otra decisión de preprocesamiento**, y cambia los resultados. Documentar qué herramienta y qué versión usaste es parte de la reproducibilidad del trabajo.

---
## 7. Formalización: un texto procesado es una lista de tokens

Después de todo el preprocesamiento, lo que queda es una estructura simple:

> **Un texto procesado = una lista ordenada de tokens (strings)**

Esta es la representación que vamos a usar como punto de partida en los próximos cuadernos, cuando necesitemos contar frecuencias, comparar documentos o construir matrices de términos.

La lista no contiene información sobre el orden original de las oraciones ni sobre relaciones sintácticas. Esa pérdida es deliberada: es el precio que pagamos por obtener una representación que se pueda operar numéricamente.

In [ ]:
# Formalización: el texto como lista de tokens
texto_procesado = pipeline_c(doc, CORRECCIONES_VOSEO)

print("REPRESENTACIÓN FINAL DEL TEXTO:")
print(f"  Tipo: {type(texto_procesado)}")
print(f"  Largo: {len(texto_procesado)} tokens")
print(f"  Contenido: {texto_procesado}")
print()
print("Esta lista es la entrada para cualquier análisis cuantitativo posterior.")

---
## 8. Ejercicio práctico

### Consigna

Tomá el siguiente texto (un fragmento adaptado de una noticia argentina) y realizá las actividades indicadas.

**Texto:**

> El Gobierno nacional anunció un nuevo paquete de medidas económicas para el sector agroindustrial. La iniciativa busca promover la exportación de productos con valor agregado y reducir la presión impositiva sobre las economías regionales. Según explicaron desde el Ministerio de Economía, las medidas entran en vigencia la semana que viene. Los representantes del campo manifestaron su apoyo pero pidieron que se incluyan también beneficios para los pequeños productores.

### Actividades

1. Procesá el texto con spaCy.
2. Construí **dos pipelines distintos** (elegí entre los que vimos o inventá el tuyo).
3. Para cada pipeline, mostrá la lista de tokens resultante y el top 10 de frecuencias.
4. **Justificá** en una celda Markdown cuál de los dos usarías para un análisis de contenido y por qué.

In [ ]:
# Texto para el ejercicio
texto_ejercicio = (
    "El Gobierno nacional anunció un nuevo paquete de medidas económicas "
    "para el sector agroindustrial. La iniciativa busca promover la "
    "exportación de productos con valor agregado y reducir la presión "
    "impositiva sobre las economías regionales. Según explicaron desde "
    "el Ministerio de Economía, las medidas entran en vigencia la semana "
    "que viene. Los representantes del campo manifestaron su apoyo pero "
    "pidieron que se incluyan también beneficios para los pequeños productores."
)

# Tu código acá
# doc_ejercicio = nlp(texto_ejercicio)
# ...

*Escribí tu justificación acá.*

---
## Cierre

En este cuaderno formalizamos algo que puede parecer trivial pero tiene consecuencias profundas: **un texto procesado es una lista de tokens, y esa lista depende de las decisiones que tomamos al construirla**.

Ideas centrales para llevarte:
- No existe un pipeline de preprocesamiento universalmente correcto.
- Cada decisión (lowercase, lematización, filtrado de stopwords) descarta información.
- Los errores del modelo (como la lematización incorrecta del voseo) se propagan silenciosamente al análisis.
- Herramientas distintas (spaCy, Stanza) producen representaciones distintas del mismo texto, y se pueden combinar vía `spacy-stanza`.
- Documentar el pipeline y la herramienta usados es tan importante como documentar los resultados.

En el próximo cuaderno vamos a usar estas listas de tokens como insumo para construir representaciones numéricas de los textos (TF-IDF).